# Week 10 — Novelty Detection, Abstention Logic & Calibration

**Learning Goals:**
- Flag out-of-distribution (OOD) inputs the model shouldn't predict on
- Calibrate prediction intervals (are 95% CI's actually covering 95%?)
- Implement an abstention policy based on uncertainty thresholds

**Deliverables:**
- OOD detection via reconstruction error (auto-encoder) or Mahalanobis distance
- Calibration curve plot
- Abstention analysis table

In [ ]:
import sys, torch
sys.path.insert(0, '../src')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.spatial.distance import mahalanobis
from scipy.stats import norm

from data_loader import load_train, load_test
from preprocess import preprocess_pipeline
from models.cnn_transformer import CNNTransformerModel
from train import compute_metrics

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device: {device}')

## 1. Load Model + Data

In [ ]:
WINDOW = 30
N_FEATURES = 14

model = CNNTransformerModel(
    n_features=N_FEATURES, seq_len=WINDOW,
    cnn_channels=64, d_model=128, nhead=4, num_layers=2, dropout=0.2
)

CHECKPOINT = '../checkpoints/cnn_transformer_best.pt'
import os
if os.path.exists(CHECKPOINT):
    state = torch.load(CHECKPOINT, map_location=device)
    model.load_state_dict(state['model_state_dict'] if 'model_state_dict' in state else state)
    print('Loaded checkpoint')
else:
    print('No checkpoint — using random weights for demo')

model.to(device).eval()

train_df = load_train('../data', 'FD001')
test_df, rul_true = load_test('../data', 'FD001')
result = preprocess_pipeline(train_df, test_df, window_size=WINDOW)
X_train, y_train = result['X_train'], result['y_train']
X_test = result['X_test']
print(f'Train: {X_train.shape}, Test: {X_test.shape}')

## 2. OOD Detection: Mahalanobis Distance

In [ ]:
# Flatten windows for Mahalanobis (simpler approach)
# Use last time-step features as a proxy
X_train_last = X_train[:, -1, :]  # (N, 14)
X_test_last = X_test[:, -1, :]

# Compute training distribution stats
mean_train = np.mean(X_train_last, axis=0)
cov_train = np.cov(X_train_last, rowvar=False)
cov_inv = np.linalg.pinv(cov_train)  # pseudo-inverse for stability

# Mahalanobis distance for each test sample
distances = np.array([
    mahalanobis(X_test_last[i], mean_train, cov_inv)
    for i in range(len(X_test_last))
])

# Also compute for train (for setting threshold)
train_distances = np.array([
    mahalanobis(X_train_last[i], mean_train, cov_inv)
    for i in range(len(X_train_last))
])

threshold_95 = np.percentile(train_distances, 95)
ood_mask = distances > threshold_95

print(f'Train Mahal. dist — mean: {train_distances.mean():.2f}, 95th: {threshold_95:.2f}')
print(f'Test OOD engines:  {ood_mask.sum()} / {len(ood_mask)} ({100*ood_mask.mean():.1f}%)')

In [ ]:
fig, ax = plt.subplots(figsize=(10, 4))
ax.hist(train_distances, bins=40, alpha=0.5, label='Train', density=True)
ax.hist(distances, bins=40, alpha=0.5, label='Test', density=True)
ax.axvline(x=threshold_95, color='red', ls='--', label=f'95th %ile = {threshold_95:.1f}')
ax.set_xlabel('Mahalanobis Distance')
ax.set_ylabel('Density')
ax.set_title('OOD Detection via Mahalanobis Distance')
ax.legend()
plt.tight_layout()
plt.savefig('../reports/ood_mahalanobis.png', dpi=150, bbox_inches='tight')
plt.show()

## 3. MC Dropout Uncertainty + Calibration

In [ ]:
# Run MC Dropout inference
N_MC = 100

X_t = torch.tensor(X_test, dtype=torch.float32).to(device)
rul_mean, rul_std, all_preds = model.predict_with_uncertainty(X_t, n_samples=N_MC)

rul_mean = rul_mean.cpu().numpy().flatten()
rul_std = rul_std.cpu().numpy().flatten()
all_preds = all_preds.cpu().numpy().squeeze()  # (N_MC, N_test)

print(f'Mean RUL range: [{rul_mean.min():.1f}, {rul_mean.max():.1f}]')
print(f'Std range:      [{rul_std.min():.2f}, {rul_std.max():.2f}]')

In [ ]:
# Calibration: check if X% prediction intervals contain X% of true values
confidence_levels = np.arange(0.1, 1.0, 0.05)
observed_coverage = []

for cl in confidence_levels:
    z = norm.ppf(0.5 + cl / 2)  # z-score for this confidence level
    lower = rul_mean - z * rul_std
    upper = rul_mean + z * rul_std
    coverage = np.mean((rul_true >= lower) & (rul_true <= upper))
    observed_coverage.append(coverage)

observed_coverage = np.array(observed_coverage)

fig, ax = plt.subplots(figsize=(6, 6))
ax.plot([0, 1], [0, 1], 'k--', label='Perfect calibration')
ax.plot(confidence_levels, observed_coverage, 'o-', color='steelblue', label='Model')
ax.fill_between(confidence_levels, confidence_levels - 0.1, confidence_levels + 0.1,
                alpha=0.1, color='gray', label='±10% band')
ax.set_xlabel('Expected Coverage')
ax.set_ylabel('Observed Coverage')
ax.set_title('Uncertainty Calibration Curve')
ax.legend()
ax.set_xlim(0, 1)
ax.set_ylim(0, 1)
plt.tight_layout()
plt.savefig('../reports/calibration_curve.png', dpi=150, bbox_inches='tight')
plt.show()

# Average Calibration Error (ACE)
ace = np.mean(np.abs(confidence_levels - observed_coverage))
print(f'Average Calibration Error: {ace:.4f}')

## 4. Abstention Policy Analysis

In [ ]:
# Sweep over uncertainty thresholds and measure MAE on non-abstained samples
thresholds = np.arange(5, 35, 2)
results = []

for t in thresholds:
    mask = rul_std <= t  # accept only low-uncertainty predictions
    n_accepted = mask.sum()
    coverage = n_accepted / len(rul_std)
    
    if n_accepted > 0:
        mae = np.mean(np.abs(rul_mean[mask] - rul_true[mask]))
    else:
        mae = np.nan
    
    results.append({'threshold': t, 'coverage': coverage, 'mae': mae, 'n_accepted': n_accepted})

df_abs = pd.DataFrame(results)
print(df_abs.to_string(index=False))

In [ ]:
fig, ax1 = plt.subplots(figsize=(10, 5))

ax1.plot(df_abs['threshold'], df_abs['mae'], 'o-', color='steelblue', label='MAE')
ax1.set_xlabel('Uncertainty Threshold (std)')
ax1.set_ylabel('MAE on Accepted Predictions', color='steelblue')

ax2 = ax1.twinx()
ax2.plot(df_abs['threshold'], df_abs['coverage'] * 100, 's--', color='darkorange', label='Coverage %')
ax2.set_ylabel('Coverage (%)', color='darkorange')

ax1.set_title('Abstention Trade-off: MAE vs Coverage')
fig.legend(loc='upper center', ncol=2, bbox_to_anchor=(0.5, 0.95))
plt.tight_layout()
plt.savefig('../reports/abstention_tradeoff.png', dpi=150, bbox_inches='tight')
plt.show()

## 5. Combined OOD + Uncertainty Decision Matrix

In [ ]:
# Decision matrix:
# In-distribution + low uncertainty  →  Trust prediction
# In-distribution + high uncertainty →  Flag for review
# Out-of-distribution               →  Abstain / escalate

UNC_THRESHOLD = 20.0

decisions = []
for i in range(len(X_test)):
    is_ood = ood_mask[i]
    high_unc = rul_std[i] > UNC_THRESHOLD
    
    if is_ood:
        decision = 'ABSTAIN (OOD)'
    elif high_unc:
        decision = 'FLAG FOR REVIEW'
    else:
        decision = 'TRUST'
    
    decisions.append({
        'engine': i + 1,
        'rul_pred': rul_mean[i],
        'rul_true': rul_true[i],
        'rul_std': rul_std[i],
        'mahal_dist': distances[i],
        'decision': decision,
    })

df_dec = pd.DataFrame(decisions)
print(df_dec['decision'].value_counts())
print(f'\nTrusted MAE: {np.mean(np.abs(df_dec[df_dec.decision=="TRUST"]["rul_pred"] - df_dec[df_dec.decision=="TRUST"]["rul_true"])):.2f}')
print(f'Overall MAE: {np.mean(np.abs(rul_mean - rul_true)):.2f}')